## PENDEKATAN 3 VADER TRANSLATION BASED (VTB)

In [1]:
import sys
!{sys.executable} -m pip install deep-translator tqdm
!{sys.executable} -m pip install ipywidgets


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
import time
import warnings
from tqdm import tqdm  
from deep_translator import GoogleTranslator

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../../outputs/data-preparation/data_preprocessing_final_no_stem.csv'
OUTPUT_DIR = '../../../outputs/sentiment-analysis/VTB'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [4]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

# Validasi kolom
required_cols = ['no', 'timestamp', 'teks', 'teks_processed']
assert all(col in df.columns for col in required_cols), f"Kolom wajib {required_cols} tidak ditemukan!"

print(f"\nData berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")
df.head()


Data berhasil dimuat: 13192 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...


In [5]:
# 2.3 Inisialisasi Translator Google (ID -> EN)
translator = GoogleTranslator(source='id', target='en')

def translate_tweet(text):
    """
    Fungsi translasi dengan proteksi error dan mekanisme retry.
    """
    if not isinstance(text, str) or pd.isna(text) or len(text.strip()) == 0:
        return ""
    
    try:
        # Percobaan pertama
        translated = translator.translate(text[:4500])
        return translated
    except Exception as e:
        # Jika gagal (biasanya karena koneksi/limit), tunggu 5 detik lalu coba sekali lagi
        print(f"\n[RETRY] Gagal pada teks: {text[:50]}... | Error: {e}")
        print("Menunggu 5 detik sebelum mencoba lagi...")
        time.sleep(5) 
        try:
            return translator.translate(text[:4500])
        except:
            # Jika tetap gagal setelah retry, kembalikan string kosong agar pipeline tidak mati
            return "" 

print("[INFO] Fungsi translate_tweet dengan mekanisme retry berhasil didefinisikan.")

[INFO] Fungsi translate_tweet dengan mekanisme retry berhasil didefinisikan.


In [6]:
# 2.4 Jalankan Translasi dengan Progress Bar dan Proteksi Rate Limit
print("\n[PROSES] Menerjemahkan teks ke Bahasa Inggris...")
print(f"Total data: {len(df)} tweet")
print("Estimasi waktu: ~20-40 menit (tergantung limit API & stabilitas internet)")

translated_texts = []

# Menggunakan enumerate untuk melacak indeks data (i)
for i, text in enumerate(tqdm(df['teks_processed'], desc="Translating")):
    # Panggil fungsi translasi
    translated_texts.append(translate_tweet(text))
    
    # 1. Proteksi Rate Limit: Jeda 1 detik setiap 50 tweet
    if (i + 1) % 50 == 0:
        time.sleep(1)
    
    # 2. Checkpoint: Simpan hasil sementara setiap 2000 tweet
    if (i + 1) % 2000 == 0:
        temp_df = df.iloc[:len(translated_texts)].copy()
        temp_df['teks_translated'] = translated_texts
        CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, f'checkpoint_{i+1}.csv')
        temp_df.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8')
        print(f"\n[CHECKPOINT] Berhasil menyimpan {i+1} data ke {CHECKPOINT_FILE}")

# Masukkan hasil akhir ke dataframe utama
df['teks_translated'] = translated_texts

print("\n[INFO] Seluruh proses translasi selesai.")


[PROSES] Menerjemahkan teks ke Bahasa Inggris...
Total data: 13192 tweet
Estimasi waktu: ~20-40 menit (tergantung limit API & stabilitas internet)


Translating:  15%|█▌        | 2000/13192 [29:41<3:50:52,  1.24s/it]


[CHECKPOINT] Berhasil menyimpan 2000 data ke ../../../outputs/sentiment-analysis/VTB\checkpoint_2000.csv


Translating:  21%|██        | 2705/13192 [40:36<2:41:59,  1.08it/s]


[RETRY] Gagal pada teks: calon kapolri komjen pol tito karnavian hari ini a... | Error: calon kapolri komjen pol tito karnavian hari ini akan ditetapkan menjadi kapolri terpilih dalam rapat paripurna DPR ke paripurnalive --> No translation was found using the current translator. Try another translator?
Menunggu 5 detik sebelum mencoba lagi...


...


[RETRY] Gagal pada teks: untuk hal yang lain kami juga menyesali sikap dan ... | Error: untuk hal yang lain kami juga menyesali sikap dan posisi partai golkar yang masih menunda RUU tindak pidana kekerasan seksual RUU TPKS yang telah disepakati oleh fraksi lainnya kamu ditindaklanjuti sebagai inisiatif DPR pada rapat BALEG DPR RI pada desember RUUPPRT RUUTPKS --> No translation was found using the current translator. Try another translator?
Menunggu 5 detik sebelum mencoba lagi...


Translating:  98%|█████████▊| 12925/13192 [4:17:06<03:56,  1.13it/s]  


[RETRY] Gagal pada teks: gara gara DPR tidak setuju RUU anggara dapat bisan... | Error: gara gara DPR tidak setuju RUU anggara dapat bisanya gedung DPR di tutup dengan dalih oposisi mau impeachment presiden jika darurat ini berhasil dapat dicontoh negara lain untuk memperpanjang jabatan ! --> No translation was found using the current translator. Try another translator?
Menunggu 5 detik sebelum mencoba lagi...


Translating: 100%|██████████| 13192/13192 [4:22:44<00:00,  1.19s/it]


[INFO] Seluruh proses translasi selesai.


In [7]:
# 2.5 Validasi Hasil Translasi
null_translated = df['teks_translated'].isna().sum()
empty_translated = (df['teks_translated'].str.strip() == '').sum()

print("\n[STATISTIK] Hasil Translasi:")
print(f"  - Total tweet          : {len(df)}")
print(f"  - Gagal/Null           : {null_translated}")
print(f"  - Kosong setelah translasi: {empty_translated}")
print(f"  - Berhasil diterjemahkan: {len(df) - null_translated - empty_translated}")

# Preview hasil
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\n🇮🇩 Processed : {df['teks_processed'].iloc[i][:80]}...")
    print(f"🇬🇧 Translated: {df['teks_translated'].iloc[i][:80]}...")


[STATISTIK] Hasil Translasi:
  - Total tweet          : 13192
  - Gagal/Null           : 0
  - Kosong setelah translasi: 3
  - Berhasil diterjemahkan: 13189

[PREVIEW] 3 Tweet Pertama:

🇮🇩 Processed : ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
🇬🇧 Translated: FAIR, for those who have state public policy, remember this! !...

🇮🇩 Processed : tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kep...
🇬🇧 Translated: Regulate online media from the DPR government, don't be sporadic, let alone sele...

🇮🇩 Processed : harus dievaluasi lagi kebijakan bebas visa terutama untuk negara tiongkok pak ! ...
🇬🇧 Translated: The visa-free policy must be evaluated again, especially for China, sir! ! hidde...


In [8]:
# Mencari baris dengan hasil translasi kosong
empty_mask = df['teks_translated'].isna() | (df['teks_translated'].str.strip() == '')
df_kosong = df[empty_mask]

print(f" Ditemukan {len(df_kosong)} baris kosong:\n")
for _, row in df_kosong.iterrows():
    print(f"No   : {row['no']}")
    print(f"Asli : {row['teks_processed'][:120]}...")
    print(f"Hasil: '{row['teks_translated']}'")
    print("-" * 60)

 Ditemukan 3 baris kosong:

No   : 6737
Asli : kom teguh saja jateng komisi dapat penugasan dari pimpinan DPR RI komisi diminta melakukan pembahasan tentang RUU...
Hasil: ''
------------------------------------------------------------
No   : 10354
Asli : RUU daerah kepulauan masuk tahap pembentukan pansus di DPR RI DPDRI...
Hasil: ''
------------------------------------------------------------
No   : 10358
Asli : nah sebagai ketua tidak perlu RUU perkebunan kan sudah ada UU perkebunan...
Hasil: ''
------------------------------------------------------------


In [9]:
# Mengisi secara manual baris yang kosong hasil translasi
df.loc[df['no'] == 6737, 'teks_translated'] = "Kom Teguh from Central Java, the commission received an assignment from the leadership of the House of Representatives, the commission was asked to conduct discussions regarding the Bill"

df.loc[df['no'] == 10354, 'teks_translated'] = "The Island Regions Bill has entered the stage of forming a special committee in the House of Representatives and Regional Representative Council"

df.loc[df['no'] == 10358, 'teks_translated'] = "As the chairman, there is no need for a Plantation Bill since the Plantation Law already exists"

# Verifikasi ulang
empty_check = df[df['teks_translated'].isna() | (df['teks_translated'] == '')]
print(f"[INFO] Sisa baris kosong: {len(empty_check)}")

[INFO] Sisa baris kosong: 0


In [10]:
# Validasi ulang untuk mencari baris yang kosong
empty_mask = df['teks_translated'].isna() | (df['teks_translated'].str.strip() == '')
df_kosong = df[empty_mask]

print(f" Ditemukan {len(df_kosong)} baris kosong:\n")
for _, row in df_kosong.iterrows():
    print(f"No   : {row['no']}")
    print(f"Asli : {row['teks_processed'][:120]}...")
    print(f"Hasil: '{row['teks_translated']}'")
    print("-" * 60)

 Ditemukan 0 baris kosong:



In [11]:
# 2.6 Simpan Data Hasil Translasi
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'data_translated.csv')
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Data berhasil disimpan ke: {OUTPUT_FILE}")


[OUTPUT] Data berhasil disimpan ke: ../../../outputs/sentiment-analysis/VTB\data_translated.csv
